### Konteks / Skenario

Platform "Kopi Nusantara" (skenario yang sama dari Tugas Mandiri Pertemuan 7) ingin mulai menyimpan **katalog menu** mereka di MongoDB — karena setiap cabang punya menu musiman/spesial dengan atribut yang berbeda-beda (tidak semua menu punya atribut yang sama, misalnya hanya menu kopi yang punya atribut "tingkat_kepahitan", sementara menu makanan punya atribut "pedas_level"). Kalian ditugaskan membangun dan menganalisis katalog ini.

**A. Membangun Katalog (Create)** *(bobot 20%)*

Buat database baru bernama `kopi_nusantara`, collection bernama `menu`. Sisipkan **minimal 12 dokumen** menu dengan field bervariasi sesuai kategorinya:
- Menu kategori **"Kopi"**: wajib memiliki field `nama_menu`, `harga`, `kategori`, dan `tingkat_kepahitan` (skala 1-5).
- Menu kategori **"Makanan"**: wajib memiliki field `nama_menu`, `harga`, `kategori`, dan `pedas_level` (skala 0-3).
- Boleh menambahkan field lain sesuai kreativitas anda (mis. `stok`, `rekomendasi_barista`).

**B. Kueri Bertingkat (Read)** *(bobot 20%)*

Tampilkan hasil dari **tiga** kueri berikut :
1. Seluruh menu kategori "Kopi" dengan `tingkat_kepahitan` di atas 3.
2. Seluruh menu dengan `harga` di antara 15000 dan 30000.
3. Menu yang **tidak memiliki** field `pedas_level` sama sekali.

**C. Pembaruan Data (Update)** *(bobot 15%)*

Tambahkan field `status` bernilai `"Tersedia"` ke seluruh dokumen menggunakan `update_many()`. Kemudian, ubah `status` menjadi `"Habis"` khusus untuk 2 menu pilihan anda menggunakan `update_one()`.

**D. Integrasi ke Spark** *(bobot 25%)*

Ambil seluruh data dari collection `menu`, ubah menjadi Spark DataFrame (ikuti pola Sub-bab 9.6 — perhatikan field yang mungkin kosong/`null` pada sebagian dokumen, seperti `tingkat_kepahitan` yang hanya dimiliki menu Kopi). Tampilkan:
1. `printSchema()` hasil konversi.
2. Rata-rata `harga` per `kategori` menggunakan `groupBy`.

**E. Kesimpulan** *(bobot 20%)*

Tulis pada markdown cell (**minimal 100 kata**): apa tantangan yang anda temui saat mengonversi data MongoDB (dengan field yang tidak seragam antar dokumen) menjadi Spark DataFrame (yang mengharuskan skema tabel seragam)? Bagaimana anda mengatasinya?

---

In [1]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["kopi_nusantara"]
menu = db["menu"]

print("Koneksi ke MongoDB berhasil!")
print("Daftar database yang ada:", client.list_database_names())

Koneksi ke MongoDB berhasil!
Daftar database yang ada: ['admin', 'config', 'kopi_nusantara', 'local', 'praktikum_bigdata']


In [2]:
#A. Membangun Katalog (Create)
# Menghapus data lama jika ada, supaya cell ini aman dijalankan ulang
menu.delete_many({})

menu.insert_many([
    # Kategori Kopi:
    {"nama_menu": "Espresso", "harga": 18000, "kategori": "Kopi", "tingkat_kepahitan": 5, "stok": 100},
    {"nama_menu": "Americano", "harga": 20000, "kategori": "Kopi", "tingkat_kepahitan": 4, "stok": 90},
    {"nama_menu": "Cappuccino", "harga": 25000, "kategori": "Kopi", "tingkat_kepahitan": 3, "stok": 80},
    {"nama_menu": "Caffe Latte", "harga": 27000, "kategori": "Kopi", "tingkat_kepahitan": 2, "stok": 80},
    {"nama_menu": "Kopi Susu Gula Aren", "harga": 22000, "kategori": "Kopi", "tingkat_kepahitan": 2, "stok": 120},
    {"nama_menu": "Kopi Tubruk", "harga": 12000, "kategori": "Kopi", "tingkat_kepahitan": 5, "stok": 70},
    {"nama_menu": "Kopi Susu", "harga": 18000, "kategori": "Kopi", "tingkat_kepahitan": 2, "stok": 50},

    # Kategori Makanan:
    {"nama_menu": "Roti Bakar", "harga": 15000, "kategori": "Makanan", "pedas_level": 0, "stok": 45},
    {"nama_menu": "Kentang Goreng", "harga": 20000, "kategori": "Makanan", "pedas_level": 0, "stok": 60},
    {"nama_menu": "Mie Goreng Pedas", "harga": 28000, "kategori": "Makanan", "pedas_level": 3, "stok": 30},
    {"nama_menu": "Singkong Goreng", "harga": 18000, "kategori": "Makanan", "pedas_level": 0, "stok": 60},
    {"nama_menu": "Pisang Goreng", "harga": 12000, "kategori": "Makanan", "pedas_level": 0, "stok": 50},
    {"nama_menu": "Nasi Goreng Spesial", "harga": 25000, "kategori": "Makanan", "pedas_level": 2, "stok": 30},
])

print("Jumlah dokumen di collection 'menu':", menu.count_documents({}))

Jumlah dokumen di collection 'menu': 13


In [3]:
#B. Kueri Bertingkat (Read)
# Kueri 1: menu Kopi dengan tingkat_kepahitan di atas 3
print("1. Menu Kopi dengan tingkat_kepahitan > 3:")
for p in menu.find({"kategori": "Kopi", "tingkat_kepahitan": {"$gt": 3}}):
    print(f"  - {p['nama_menu']} (kepahitan={p['tingkat_kepahitan']}, Rp{p['harga']:,})")

print()

# Kueri 2: menu dengan harga di antara 15000 dan 30000
print("2. Menu dengan harga antara 15000 dan 30000:")
for p in menu.find({"harga": {"$gte": 15000, "$lte": 30000}}):
    print(f"  - {p['nama_menu']} (Rp{p['harga']:,})")

print()

# Kueri 3: menu yang tidak memiliki field pedas_level sama sekali
print("3. Menu tanpa field pedas_level:")
for p in menu.find({"pedas_level": {"$exists": False}}):
    print(f"  - {p['nama_menu']} ({p['kategori']})")

1. Menu Kopi dengan tingkat_kepahitan > 3:
  - Espresso (kepahitan=5, Rp18,000)
  - Americano (kepahitan=4, Rp20,000)
  - Kopi Tubruk (kepahitan=5, Rp12,000)

2. Menu dengan harga antara 15000 dan 30000:
  - Espresso (Rp18,000)
  - Americano (Rp20,000)
  - Cappuccino (Rp25,000)
  - Caffe Latte (Rp27,000)
  - Kopi Susu Gula Aren (Rp22,000)
  - Kopi Susu (Rp18,000)
  - Roti Bakar (Rp15,000)
  - Kentang Goreng (Rp20,000)
  - Mie Goreng Pedas (Rp28,000)
  - Singkong Goreng (Rp18,000)
  - Nasi Goreng Spesial (Rp25,000)

3. Menu tanpa field pedas_level:
  - Espresso (Kopi)
  - Americano (Kopi)
  - Cappuccino (Kopi)
  - Caffe Latte (Kopi)
  - Kopi Susu Gula Aren (Kopi)
  - Kopi Tubruk (Kopi)
  - Kopi Susu (Kopi)


In [4]:
#C. Pembaruan Data (Update)
# update_many() — status "Tersedia" untuk seluruh dokumen
menu.update_many({}, {"$set": {"status": "Tersedia"}})

# update_one() — status "Habis" untuk 2 menu pilihan
menu.update_one({"nama_menu": "Espresso"}, {"$set": {"status": "Habis"}})
menu.update_one({"nama_menu": "Pisang Goreng"}, {"$set": {"status": "Habis"}})

print("Setelah update:")
for p in menu.find({}, {"_id": 0, "nama_menu": 1, "status": 1}):
    print(f"  - {p['nama_menu']}: {p['status']}")

Setelah update:
  - Espresso: Habis
  - Americano: Tersedia
  - Cappuccino: Tersedia
  - Caffe Latte: Tersedia
  - Kopi Susu Gula Aren: Tersedia
  - Kopi Tubruk: Tersedia
  - Kopi Susu: Tersedia
  - Roti Bakar: Tersedia
  - Kentang Goreng: Tersedia
  - Mie Goreng Pedas: Tersedia
  - Singkong Goreng: Tersedia
  - Pisang Goreng: Habis
  - Nasi Goreng Spesial: Tersedia


In [5]:
#D. Integrasi ke Spark
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, count
import pandas as pd

spark = SparkSession.builder \
    .appName("Tugas9-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Langkah 1: Ambil seluruh dokumen dari MongoDB
dokumen_mongo = list(menu.find({}))

# Langkah 2: Ubah menjadi pandas DataFrame
df_pandas = pd.DataFrame(dokumen_mongo)

# Langkah 3: Sesuaikan tipe data agar terbaca Spark
df_pandas["_id"] = df_pandas["_id"].astype(str)

# Kolom yang hanya dimiliki sebagian dokumen dijadikan Int64 agar tetap integer
for kol in ["tingkat_kepahitan", "pedas_level"]:
    df_pandas[kol] = df_pandas[kol].astype("Int64")

# Nilai kosong (NaN/NA) diganti None supaya menjadi null di Spark
df_pandas = df_pandas.astype(object).where(df_pandas.notnull(), None)

# Langkah 4: Konversi ke Spark DataFrame
df_spark_menu = spark.createDataFrame(df_pandas)

print("1. printSchema():")
df_spark_menu.printSchema()
df_spark_menu.show()

print("2. Rata-rata harga per kategori:")
df_spark_menu.groupBy("kategori").agg(
    avg("harga").alias("rata_rata_harga"),
    count("nama_menu").alias("jumlah_menu")
).show()

spark.stop()
print("SparkSession ditutup.")

26/11/04 04:27:15 WARN Utils: Your hostname, yan resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/11/04 04:27:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/04 04:27:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


1. printSchema():
root
 |-- _id: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- tingkat_kepahitan: long (nullable = true)
 |-- stok: long (nullable = true)
 |-- status: string (nullable = true)
 |-- pedas_level: long (nullable = true)



+--------------------+-------------------+-----+--------+-----------------+----+--------+-----------+
|                 _id|          nama_menu|harga|kategori|tingkat_kepahitan|stok|  status|pedas_level|
+--------------------+-------------------+-----+--------+-----------------+----+--------+-----------+
|6aeab4215bc851b7c...|           Espresso|18000|    Kopi|                5| 100|   Habis|       NULL|
|6aeab4215bc851b7c...|          Americano|20000|    Kopi|                4|  90|Tersedia|       NULL|
|6aeab4215bc851b7c...|         Cappuccino|25000|    Kopi|                3|  80|Tersedia|       NULL|
|6aeab4215bc851b7c...|        Caffe Latte|27000|    Kopi|                2|  80|Tersedia|       NULL|
|6aeab4215bc851b7c...|Kopi Susu Gula Aren|22000|    Kopi|                2| 120|Tersedia|       NULL|
|6aeab4215bc851b7c...|        Kopi Tubruk|12000|    Kopi|                5|  70|Tersedia|       NULL|
|6aeab4215bc851b7c...|          Kopi Susu|18000|    Kopi|                2|  50|Te

#E. Kesimpulan
Tantangan utama saat mengonversi data MongoDB ke Spark DataFrame adalah ketidakseragaman field antar dokumen. Menu Kopi memiliki field tingkat_kepahitan tetapi tidak memiliki pedas_level, sedangkan menu Makanan sebaliknya. Spark DataFrame membutuhkan skema tabel yang seragam, sehingga setiap kolom harus ada untuk semua baris. Tantangan lainnya adalah tipe data khusus MongoDB, yaitu _id bertipe ObjectId yang tidak dikenali Spark, serta pandas yang mengubah kolom bilangan bulat berisi nilai kosong menjadi float dengan NaN, bukan null. Cara mengatasinya: pertama, _id diubah menjadi string dengan astype(str). Kedua, kolom tingkat_kepahitan dan pedas_level diubah ke tipe Int64 pandas, lalu seluruh nilai kosong diganti None agar menjadi null di Spark. Hasilnya, field yang tidak dimiliki suatu dokumen otomatis menjadi null pada kolom yang sama, sehingga skema menjadi seragam dan analisis seperti groupBy dapat dijalankan tanpa error.